# LLaMA-3-8B Fine-Tuning for Ingredient Safety Classification

This notebook fine-tunes LLaMA-3-8B-Instruct on ingredient safety data using Unsloth (4-bit QLoRA).

**Dataset**: `ingredient_safety_unique.jsonl` (6,184 samples)

**GPU Requirement**: T4 (15GB) or better - Google Colab Free tier works!

## Step 1: Install Unsloth & Dependencies

In [ ]:
# Install Unsloth (Colab-optimized with 4-bit quantization)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## Step 2: Mount Google Drive & Upload Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory structure
!mkdir -p /content/ingredient_decoder/data
!mkdir -p /content/ingredient_decoder/models

### Upload Your Dataset

Copy your JSONL file to Colab. Choose ONE of these options:

In [ ]:
# OPTION 1: Copy from Google Drive (if already uploaded)
# !cp "/content/drive/MyDrive/ingredient_safety_unique.jsonl" /content/ingredient_decoder/data/

# OPTION 2: Upload directly via Colab file uploader
from google.colab import files
uploaded = files.upload()  # Select ingredient_safety_unique.jsonl from your local machine

# Move uploaded file to data directory
import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"/content/ingredient_decoder/data/{filename}")

print(f"Uploaded files: {list(uploaded.keys())}")

## Step 3: Load & Inspect Dataset

In [ ]:
from datasets import load_dataset
import json

# Load the JSONL dataset
dataset_path = "/content/ingredient_decoder/data/ingredient_safety_unique.jsonl"
dataset = load_dataset('json', data_files=dataset_path, split='train')

print(f"Total samples: {len(dataset)}")
print(f"\nSample entry:")
print(json.dumps(dataset[0], indent=2))

# Count categories
from collections import Counter
categories = [ex['output'].split('\n')[0].replace('Risk Level: ', '') for ex in dataset]
print(f"\nCategory distribution:")
for cat, count in Counter(categories).items():
    print(f"  {cat}: {count} ({count/len(dataset)*100:.1f}%)")

## Step 4: Load LLaMA-3-8B with Unsloth (4-bit Quantization)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Model configuration
MODEL_NAME = "unsloth/Llama-3-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512  # Reduced for memory efficiency
LOAD_IN_4BIT = True

# Load model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    fast_inference=True,  # Enable Flash Attention
)

print("Model loaded successfully!")
print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")

## Step 5: Configure LoRA Adapters

In [ ]:
# Configure LoRA for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (higher = more parameters, better quality)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory-efficient training
    random_state=42,
)

print("LoRA adapters configured!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 6: Format Dataset for Instruction Tuning

In [ ]:
# LLaMA-3 instruction template
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Add EOS token for proper generation

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = ALPACA_PROMPT.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Formatted sample:\n{dataset[0]['text'][:300]}...")

## Step 7: Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training configuration (optimized for Colab T4)
BATCH_SIZE = 2  # Small batch for 4GB VRAM
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 2 * 4 = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=10,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    output_dir="/content/ingredient_decoder/models/output",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",  # Disable wandb reporting
)

print("Training arguments configured!")

## Step 8: Initialize Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # Set True for faster training if sequences are short
    args=training_args,
)

print("Trainer initialized!")
print(f"GPU Memory before training: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")

## Step 9: Start Training

In [ ]:
# Start training
print("Starting training... This may take 30-60 minutes on Colab T4.")
trainer_stats = trainer.train()

print(f"\nTraining completed!")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Training loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 10: Test the Fine-Tuned Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test with sample ingredients
test_ingredients = ["citric acid", "sodium benzoate", "high fructose corn syrup"]

for ingredient in test_ingredients:
    prompt = ALPACA_PROMPT.format(
        "Analyze this food ingredient and classify its safety level as Safe, Moderate, or Harmful. Provide a brief explanation.",
        ingredient,
        ""
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*50}")
    print(f"Ingredient: {ingredient}")
    print(f"{'='*50}")
    print(response.split("### Response:")[-1].strip())

## Step 11: Save Model to Google Drive

In [ ]:
# Save the fine-tuned adapter to Google Drive
SAVE_PATH = "/content/drive/MyDrive/ingredient_decoder_models/fine_tuned_adapter"
!mkdir -p {SAVE_PATH}

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to: {SAVE_PATH}")

# Also save a copy in the Colab runtime (for immediate download)
local_save_path = "/content/ingredient_decoder/models/fine_tuned_adapter"
!mkdir -p {local_save_path}
model.save_pretrained(local_save_path)
tokenizer.save_pretrained(local_save_path)
print(f"Local copy saved to: {local_save_path}")

## Step 12: Download Model (Optional)

In [ ]:
# Download the model as a zip file
import shutil

shutil.make_archive('/content/fine_tuned_adapter', 'zip', local_save_path)
files.download('/content/fine_tuned_adapter.zip')
print("Download started!")

## Training Complete!

### Next Steps:
1. **Model saved to Google Drive**: `/content/drive/MyDrive/ingredient_decoder_models/fine_tuned_adapter`
2. **Download locally** (optional): Use the download cell above
3. **Use the model**: Load the adapter with the base LLaMA-3-8B model for inference

### Loading the Fine-Tuned Model Later:
```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3-8B-Instruct-bnb-4bit",
    adapter_name="/path/to/fine_tuned_adapter",
    load_in_4bit=True,
)
```